# Comparação do pipeline: aorta e óstios

Avaliação operacional do **novo padrão: filtro + envelope + lower100/pad2**,
comparado ao P99.9 puro e à correção da aorta sem padding. Mantém -300 HU,
P99.9, RG e o pós-processamento arterial. O filtro usa geometria 4.8/8 mm,
cinco círculos sintéticos, envelope 2.25r/margem 10 e level set b0.6/r0.10/i26.

A tabela principal usa treino (30), validação (270) e teste (700), sem juntar
subconjuntos. A revisão visual histórica (30/60) aparece antes da visão geral:
sucesso automático dos óstios não equivale a qualidade visual da aorta.
Nenhum pipeline pesado ou HTML é gerado aqui; os dados vêm dos runs salvos.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Localiza a raiz antes de importar os módulos do projeto.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_visual_reviews,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402
from utils.project.results import normalize_ostia_status  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)
from utils.comparison_utils.paired_statistics import (  # noqa: E402
    adjust_holm,
    compare_paired_dice,
)

## 1. Runs e coortes

Caminhos explícitos: não seleciona silenciosamente o run mais recente.


In [ ]:
RUNS = {
    "train": {
        "puro": "output/segmentation/runs/mid_res/current_baseline_p99_9/train/2026-08-06_18-43-37",
        "filtro_envelope": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/train/trajectory_geometry_r4_8_c8_0_p99_9_m300/2026-09-05_19-24-41",
        "filtro_envelope_pad2": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/train/ostia_localization/lower100_pad2/2026-09-06_08-08-25",
    },
    "val": {
        "puro": "output/segmentation/runs/mid_res/current_baseline_p99_9/val/2026-08-06_22-43-14",
        "filtro_envelope": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/val/ostia_validation270/baseline_pad0/2026-09-07_07-15-02",
        "filtro_envelope_pad2": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/val/ostia_validation270/lower100_pad2/2026-09-07_07-15-15",
    },
    "test": {
        "puro": "output/segmentation/runs/mid_res/current_baseline_p99_9/test/2026-08-06_10-04-22",
        "filtro_envelope": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/test/ostia_comparison/baseline_pad0/2026-09-07_10-37-23",
        "filtro_envelope_pad2": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/test/ostia_comparison/lower100_pad2/2026-09-07_10-37-38",
    },
}
EXPECTED_IMAGES = {"train": 30, "val": 270, "test": 700}
COHORT_NAMES = {"train": "Treino", "val": "Validação", "test": "Teste"}
METHOD_NAMES = {
    "puro": "P99.9 puro (histórico)",
    "filtro_envelope": "Filtro + envelope / pad0",
    "filtro_envelope_pad2": "Filtro + envelope / pad2 (padrão)",
}
SUCCESS = {"both_correct", "both_tolerable"}
run_frames = {}
for split, variants in RUNS.items():
    for variant, directory in variants.items():
        path = REPO_ROOT / directory / "numeric" / f"results_{split}.csv"
        frame = pd.read_csv(path)
        frame["IMG_ID"] = pd.to_numeric(frame["IMG_ID"], errors="raise").astype(int)
        if frame["IMG_ID"].duplicated().any() or len(frame) != EXPECTED_IMAGES[split]:
            raise ValueError(f"Coorte incompleta/duplicada: {split}/{variant}")
        status = frame["ostia_detection_status"].map(normalize_ostia_status)
        frame["ostia_success"] = status.isin(SUCCESS)
        frame["artery_dice"] = pd.to_numeric(frame["artery_dice"], errors="raise")
        if not frame["artery_dice"].between(0, 1).all():
            raise ValueError(f"Dice inválido/ausente: {split}/{variant}")
        run_frames[split, variant] = frame
    ids = set(run_frames[split, "puro"]["IMG_ID"])
    if any(set(run_frames[split, v]["IMG_ID"]) != ids for v in variants):
        raise ValueError(f"IDs diferentes em {split}")

## 2. Resultados visuais: 30 treino e 60 validação

Esta seção vem antes da visão geral para contextualizar visualmente as mudanças da aorta. A classificação **aorta boa/ruim** vem das coortes revisadas manualmente de 30 imagens de treino e 60 de validação. A revisão refinada `b0.6/r0.10/i26`, que elevou a validação de 52/60 para 56/60 aortas boas, representa a etapa de aorta comum ao pad0 e ao pad2.

Para manter consistência entre os gráficos, **Dice e sucesso dos óstios são calculados a partir dos mesmos runs atuais usados na visão geral**, restringindo a validação aos 60 IDs revisados. Assim, o filtro + envelope corresponde ao pad0 de referência do pad2.

O pad2 modifica a referência em dois parâmetros da seleção dos óstios: `lower_fraction` passa de `0.85` para `1.0` e `surface_padding_radius` passa de `0` para `2`. Os demais elementos desta comparação permanecem associados à configuração refinada da aorta.

In [ ]:
# A revisão manual fornece somente a classificação visual da aorta.
# Dice e óstios são sempre lidos dos mesmos runs usados na visão geral.
review_catalog = load_aorta_visual_reviews(
    REPO_ROOT / "config/aorta_visual_reviews.json"
)
review_variants = (
    ("normal", "puro", "Puro"),
    ("levelset_b0_6_r0_10_i26", "filtro_envelope", "Filtro + envelope / pad0"),
    (
        "levelset_b0_6_r0_10_i26",
        "filtro_envelope_pad2",
        "Filtro + envelope / pad2",
    ),
)

visual_rows = []
for split in ("train", "val"):
    for review_variant, run_variant, label in review_variants:
        review = get_aorta_visual_review(review_catalog, review_variant, split)
        good, bad = review["aorta_good_ids"], review["aorta_bad_ids"]
        reviewed_ids = good | bad

        # Recorta o run atual exatamente para a coorte inspecionada visualmente.
        summary = run_frames[split, run_variant]
        summary = summary[summary["IMG_ID"].isin(reviewed_ids)].copy()
        if set(summary["IMG_ID"]) != reviewed_ids:
            missing = sorted(reviewed_ids.difference(summary["IMG_ID"]))
            raise ValueError(
                f"IDs revisados ausentes no run atual: {run_variant}/{split}: {missing}"
            )

        ostia_success = summary["ostia_success"]
        dice = summary["artery_dice"]
        visual_rows.append(
            {
                "Conjunto": COHORT_NAMES[split],
                "Versão": label,
                "N": len(summary),
                "Aortas boas": len(good),
                "Aortas boas (%)": 100 * len(good) / len(reviewed_ids),
                "Óstios OK": int(ostia_success.sum()),
                "Óstios OK (%)": 100 * ostia_success.mean(),
                "Dice": dice.mean(),
            }
        )

visual_metrics = pd.DataFrame(visual_rows)
display(visual_metrics.round(4))

# Separa as escalas: porcentagens não compartilham eixo com Dice.
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
colors = ["#4C78A8", "#54A24B", "#E45756"]
metrics = (
    ("Aortas boas (%)", "Qualidade visual da aorta (%)", (0, 108), ".1f"),
    ("Óstios OK (%)", "Sucesso dos óstios (%)", (0, 108), ".1f"),
    ("Dice", "Dice médio", (0, 1.05), ".3f"),
)

for row, split in enumerate(("train", "val")):
    data = visual_metrics[visual_metrics["Conjunto"].eq(COHORT_NAMES[split])]
    for column, (metric, title, limits, value_format) in enumerate(metrics):
        axis = axes[row, column]
        bars = axis.bar(data["Versão"], data[metric], color=colors)
        axis.set_title(f"{COHORT_NAMES[split]}: {title}", fontsize=11)
        axis.set_ylim(*limits)
        axis.tick_params(axis="x", rotation=15, labelsize=9)
        axis.grid(axis="y", alpha=0.25)
        axis.bar_label(bars, fmt=f"%{value_format}", padding=3, fontsize=9)

plt.show()

## 3. Desempenho geral

Dice inclui todas as falhas. Óstios corretos e toleráveis contam como sucesso.


In [ ]:
# Preserva Dice zero e falhas na média geral.
rows = []
for (split, variant), frame in run_frames.items():
    valid = frame["ostia_success"]
    rows.append(
        {
            "subconjunto": COHORT_NAMES[split],
            "variante": METHOD_NAMES[variant],
            "exames": len(frame),
            "óstios_sucesso": int(valid.sum()),
            "óstios_sucesso_%": 100 * valid.mean(),
            "dice_médio_todos": frame["artery_dice"].mean(),
            "dice_mediano": frame["artery_dice"].median(),
            "dice_std": frame["artery_dice"].std(),
            "dice_médio_óstios_válidos": frame.loc[valid, "artery_dice"].mean(),
            "dice_zero": int(frame["artery_dice"].eq(0).sum()),
        }
    )
pipeline_overview = pd.DataFrame(rows)
# Mantém as métricas completas em pipeline_overview, mas exibe só o essencial.
overview_view = pipeline_overview[
    [
        "subconjunto",
        "variante",
        "exames",
        "dice_médio_todos",
        "óstios_sucesso",
        "óstios_sucesso_%",
    ]
].copy()
overview_view["variante"] = overview_view["variante"].replace(
    {
        METHOD_NAMES["puro"]: "Puro",
        METHOD_NAMES["filtro_envelope"]: "Filtro + envelope",
        METHOD_NAMES["filtro_envelope_pad2"]: "Filtro + envelope + pad2",
    }
)
display(
    overview_view.rename(
        columns={
            "subconjunto": "Conjunto",
            "variante": "Método",
            "exames": "N",
            "dice_médio_todos": "Dice",
            "óstios_sucesso": "Óstios OK",
            "óstios_sucesso_%": "Óstios (%)",
        }
    ).round(4)
)

## 4. Métricas por conjunto

Os painéis apresentam Dice médio e sucesso dos óstios para as coortes completas de treino, validação e teste.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 11), constrained_layout=True)
for row, split in enumerate(RUNS):
    data = pipeline_overview[pipeline_overview["subconjunto"].eq(COHORT_NAMES[split])]
    for col, (metric, label, limit) in enumerate(
        (
            ("dice_médio_todos", "Dice médio", 1.1),
            ("óstios_sucesso_%", "Sucesso dos óstios (%)", 110),
        )
    ):
        ax = axes[row, col]
        bars = ax.bar(
            ["Puro", "Filtro + envelope", "+ pad2"],
            data[metric],
            color=["#4C78A8", "#54A24B", "#E45756"],
        )
        ax.set_title(COHORT_NAMES[split])
        ax.set_ylabel(label)
        ax.set_ylim(0, limit)
        ax.bar_label(bars, fmt="%.3f" if col == 0 else "%.1f", padding=4)
        ax.grid(axis="y", alpha=0.2)
plt.show()

## 5. Teste pareado de Wilcoxon

**Δ Dice = pad2 − referência**; positivo favorece pad2. `W` é a estatística
do teste de Wilcoxon bilateral: a menor soma dos postos positivos e negativos
das diferenças não nulas. Seu valor depende do número de pares e deve ser
interpretado junto com o p-valor, não como tamanho da melhora.

O efeito rank-biserial compara os postos favoráveis e contrários ao pad2.
Ele varia de -1 a 1: valores positivos favorecem pad2, valores negativos
favorecem a referência e valores próximos de zero indicam equilíbrio. Ele
resume direção e dominância das diferenças, mas não representa ganho médio de Dice.

Wilcoxon bilateral sobre os mesmos exames, com deltas arredondados a 12 casas
e deltas zero fora dos ranks. Holm permanece aplicado às nove comparações
originais, incluindo puro versus pad0. A tabela mostra apenas as seis que
envolvem o novo padrão; os detalhes permanecem em `paired_statistics`.

O desempenho de cada método é apresentado como mediana [Q1–Q3], coerente
com a análise não paramétrica. O IC95% corresponde ao Δ Dice médio e é
estimado por bootstrap pareado com 10.000 reamostragens e semente fixa 42.
A coluna `melhorou/piorou/igual` complementa o resultado agregado com a
direção observada exame a exame.

`p (Holm) < 0,05` indica diferença estatística após ajuste, não necessariamente
melhora clinicamente relevante. O teste não avalia diretamente a média nem
confirma a qualidade visual da aorta.


In [ ]:
# A família inclui três comparações por split; Holm é aplicado às nove.
pair_rows = []
for split in RUNS:
    for reference, candidate in (
        ("puro", "filtro_envelope"),
        ("puro", "filtro_envelope_pad2"),
        ("filtro_envelope", "filtro_envelope_pad2"),
    ):
        ref = run_frames[split, reference]
        cand = run_frames[split, candidate]
        stats = compare_paired_dice(ref, cand)
        ostia = (
            ref.set_index("IMG_ID")["ostia_success"]
            .to_frame("reference")
            .join(
                cand.set_index("IMG_ID")["ostia_success"].rename("candidate"),
                how="inner",
            )
        )
        pair_rows.append(
            {
                "subconjunto": COHORT_NAMES[split],
                "referência": METHOD_NAMES[reference],
                "candidato": METHOD_NAMES[candidate],
                **stats,
                "óstios_recuperados": int(
                    (~ostia["reference"] & ostia["candidate"]).sum()
                ),
                "óstios_perdidos": int(
                    (ostia["reference"] & ~ostia["candidate"]).sum()
                ),
            }
        )
paired_statistics = pd.DataFrame(pair_rows)
paired_statistics["p_holm"] = adjust_holm(paired_statistics["p_value"])
paired_statistics["significativo_005"] = paired_statistics["p_holm"].lt(0.05)

# Formata desempenho e intervalo sem descartar os valores numéricos originais.
paired_statistics["desempenho_referência"] = paired_statistics.apply(
    lambda row: (
        f"{row['baseline_median_dice']:.3f} "
        f"[{row['baseline_q1_dice']:.3f}–{row['baseline_q3_dice']:.3f}]"
    ),
    axis=1,
)
paired_statistics["desempenho_candidato"] = paired_statistics.apply(
    lambda row: (
        f"{row['candidate_median_dice']:.3f} "
        f"[{row['candidate_q1_dice']:.3f}–{row['candidate_q3_dice']:.3f}]"
    ),
    axis=1,
)
paired_statistics["ic95_delta"] = paired_statistics.apply(
    lambda row: (
        f"[{row['mean_delta_ci_95_low']:.4f}; {row['mean_delta_ci_95_high']:.4f}]"
    ),
    axis=1,
)
paired_statistics["desfechos"] = paired_statistics.apply(
    lambda row: (
        f"{int(row['improved_images'])}/"
        f"{int(row['worse_images'])}/"
        f"{int(row['unchanged_images'])}"
    ),
    axis=1,
)

# Reduz a largura sem mudar testes, pares, p-valores ou família de Holm.
wilcoxon_view = paired_statistics.loc[
    paired_statistics["candidato"].eq(METHOD_NAMES["filtro_envelope_pad2"]),
    [
        "subconjunto",
        "referência",
        "paired_images",
        "desempenho_referência",
        "desempenho_candidato",
        "mean_delta_dice",
        "ic95_delta",
        "wilcoxon_statistic",
        # "rank_biserial_effect",
        "p_value",
        # "p_holm",
        "desfechos",
        "significativo_005",
    ],
].copy()
wilcoxon_view["referência"] = wilcoxon_view["referência"].replace(
    {
        METHOD_NAMES["puro"]: "Pad2 − puro",
        METHOD_NAMES["filtro_envelope"]: "Pad2 − filtro/envelope",
    }
)
wilcoxon_view["significativo_005"] = wilcoxon_view["significativo_005"].map(
    {
        True: "Diferença detectada",
        False: "Não significativa",
    }
)
wilcoxon_view = wilcoxon_view.rename(
    columns={
        "subconjunto": "Conjunto",
        "referência": "Comparação",
        "paired_images": "N",
        "desempenho_referência": "Referência: mediana [IQR]",
        "desempenho_candidato": "Pad2: mediana [IQR]",
        "mean_delta_dice": "Δ Dice médio",
        "ic95_delta": "IC95% do Δ médio",
        "wilcoxon_statistic": "W",
        # "rank_biserial_effect": "r rank-biserial",
        "p_value": "p bruto",
        # "p_holm": "p (Holm)",
        "desfechos": "Melhorou/piorou/igual",
        "significativo_005": "Conclusão",
    }
)
with pd.option_context("display.float_format", "{:.4g}".format):
    display(wilcoxon_view)